# Final Assam GNN + GRU - Colab Ready

This notebook is built to run **after** your two Assam notebooks:

1. `Assam_preprocessing.ipynb`
2. `Assam_Cnn.ipynb`

It does not redo Sentinel-1 or DEM preprocessing. It reuses the patch files created by preprocessing:

```text
/content/drive/MyDrive/Prj_3_Data/Assam_Model_Ready/<year>/patches/
  X_pre_0.npy
  X_flood_0.npy
  X_dem_0.npy
  y_mask_0.npy
  X_pre_1.npy
  X_flood_1.npy
  X_dem_1.npy
  y_mask_1.npy
```

It also uses the full quick masks:

```text
/content/drive/MyDrive/Prj_3_Data/Assam_Model_Ready/<year>/Assam_<year>_quick_flood_mask.npy
```

Those full masks let this notebook reconstruct the original patch grid coordinates exactly, instead of using approximate coordinates.

Final model:

```text
Sentinel-1 + DEM patch statistics -> GCN branch
Rainfall + river temporal sequence -> GRU branch
GCN embedding + GRU embedding      -> flood/non-flood patch classifier
```

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib tqdm

## 2. Mount Drive and Auto-Detect Project Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

CANDIDATE_DRIVE_ROOTS = [
    Path('/content/drive/MyDrive/Prj_3_Data'),
    Path('/content/drive/MyDrive/PRJ3_DATA'),
]

MODEL_READY_ROOT = None
DRIVE_ROOT = None
for root in CANDIDATE_DRIVE_ROOTS:
    candidate = root / 'Assam_Model_Ready'
    if candidate.exists():
        DRIVE_ROOT = root
        MODEL_READY_ROOT = candidate
        break

if MODEL_READY_ROOT is None:
    raise FileNotFoundError(
        'Could not find Assam_Model_Ready under /content/drive/MyDrive/Prj_3_Data or /content/drive/MyDrive/PRJ3_DATA. '
        'Run Assam_preprocessing.ipynb first.'
    )

TABULAR_DATA_CANDIDATES = [
    DRIVE_ROOT / 'data',
    Path('/content/drive/MyDrive/Prj_3_Data/data'),
    Path('/content/drive/MyDrive/PRJ3_DATA/data'),
    DRIVE_ROOT,
]

OUTPUT_ROOT = DRIVE_ROOT / 'Assam_GNN_GRU_Outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023]
PATCH_SIZE = 128
STRIDE = 128
CHUNK_SIZE = 2000

print('DRIVE_ROOT:', DRIVE_ROOT)
print('MODEL_READY_ROOT:', MODEL_READY_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('Tabular search roots:')
for p in TABULAR_DATA_CANDIDATES:
    print(' ', p, 'exists=', p.exists())

## 3. Imports and Reproducibility

In [ ]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)

## 4. Assam Event Windows

These dates match the active event windows inside `Assam_preprocessing.ipynb`, not the older generic pipeline dates.

In [ ]:
EVENT_DATES = {
    2019: ('2019-07-20', '2019-08-20'),
    2020: ('2020-07-15', '2020-08-25'),
    2021: ('2021-07-10', '2021-08-20'),
    2022: ('2022-07-05', '2022-08-10'),
    2023: ('2023-07-15', '2023-08-25'),
}

## 5. Load Assam Rainfall Features

In [ ]:
def find_existing_file(names):
    for root in TABULAR_DATA_CANDIDATES:
        for name in names:
            p = root / name
            if p.exists():
                return p
    return None

def find_rainfall_file(year):
    return find_existing_file([f'Assam_{year}.csv', f'assam_{year}.csv'])

def load_rainfall_features(years):
    rows = []
    for year in years:
        path = find_rainfall_file(year)
        if path is None:
            print(f'Rainfall missing for {year}; filling zeros')
            rows.append({'year': year})
            continue

        print(f'Rainfall {year}:', path)
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['Avg_rainfall'] = pd.to_numeric(df['Avg_rainfall'], errors='coerce')
        df = df.dropna(subset=['Date', 'Avg_rainfall'])

        event_start = pd.Timestamp(EVENT_DATES[year][0])
        event_end = pd.Timestamp(EVENT_DATES[year][1])
        pre_start = event_start - pd.Timedelta(days=30)
        post_end = event_end + pd.Timedelta(days=30)

        pre = df[(df['Date'] >= pre_start) & (df['Date'] < event_start)]
        event = df[(df['Date'] >= event_start) & (df['Date'] <= event_end)]
        post = df[(df['Date'] > event_end) & (df['Date'] <= post_end)]
        district_event = event.groupby('District')['Avg_rainfall'].sum() if 'District' in event.columns else pd.Series(dtype=float)

        rows.append({
            'year': year,
            'rain_pre30_mean': pre['Avg_rainfall'].mean(),
            'rain_pre30_sum': pre['Avg_rainfall'].sum(),
            'rain_event_mean': event['Avg_rainfall'].mean(),
            'rain_event_sum': event['Avg_rainfall'].sum(),
            'rain_event_max': event['Avg_rainfall'].max(),
            'rain_post30_mean': post['Avg_rainfall'].mean(),
            'rain_post30_sum': post['Avg_rainfall'].sum(),
            'rain_district_event_sum_mean': district_event.mean() if len(district_event) else np.nan,
            'rain_district_event_sum_max': district_event.max() if len(district_event) else np.nan,
            'rain_rows': len(df),
        })
    return pd.DataFrame(rows).fillna(0.0)

rain_features = load_rainfall_features(YEARS)
rain_features

## 6. Load Assam River-Level Features

In [ ]:
RIVER_TIME_COL = 'Data Acquisition Time'
RIVER_LEVEL_COL = 'River Water Level Telemetry Hourly (meter)'

def find_river_file():
    return find_existing_file(['Assam_Riverlevel.csv', 'assam_riverlevel.csv'])

def load_river_features(years):
    path = find_river_file()
    if path is None:
        print('River file missing; filling zeros')
        return pd.DataFrame({'year': years})

    print('River file:', path)
    df = pd.read_csv(path)
    df[RIVER_TIME_COL] = pd.to_datetime(df[RIVER_TIME_COL], errors='coerce', dayfirst=True)
    df[RIVER_LEVEL_COL] = pd.to_numeric(df[RIVER_LEVEL_COL], errors='coerce')
    df = df.dropna(subset=[RIVER_TIME_COL, RIVER_LEVEL_COL])

    rows = []
    for year in years:
        event_start = pd.Timestamp(EVENT_DATES[year][0])
        event_end = pd.Timestamp(EVENT_DATES[year][1])
        pre_start = event_start - pd.Timedelta(days=30)
        post_end = event_end + pd.Timedelta(days=30)

        pre = df[(df[RIVER_TIME_COL] >= pre_start) & (df[RIVER_TIME_COL] < event_start)]
        event = df[(df[RIVER_TIME_COL] >= event_start) & (df[RIVER_TIME_COL] <= event_end + pd.Timedelta(days=1))]
        post = df[(df[RIVER_TIME_COL] > event_end) & (df[RIVER_TIME_COL] <= post_end)]
        station_event = event.groupby('Station')[RIVER_LEVEL_COL].max() if 'Station' in event.columns else pd.Series(dtype=float)

        rows.append({
            'year': year,
            'river_pre30_mean': pre[RIVER_LEVEL_COL].mean(),
            'river_pre30_max': pre[RIVER_LEVEL_COL].max(),
            'river_event_mean': event[RIVER_LEVEL_COL].mean(),
            'river_event_max': event[RIVER_LEVEL_COL].max(),
            'river_post30_mean': post[RIVER_LEVEL_COL].mean(),
            'river_post30_max': post[RIVER_LEVEL_COL].max(),
            'river_station_event_max_mean': station_event.mean() if len(station_event) else np.nan,
            'river_station_event_max_max': station_event.max() if len(station_event) else np.nan,
            'river_event_station_count': event['Station'].nunique() if 'Station' in event.columns else 0,
            'river_rows': len(df),
        })
    return pd.DataFrame(rows).fillna(0.0)

river_features = load_river_features(YEARS)
river_features

## 7. Verify Patch Files From Assam Preprocessing

In [ ]:
def get_patch_paths(year):
    patch_dir = MODEL_READY_ROOT / str(year) / 'patches'
    return {
        'patch_dir': patch_dir,
        'pre': sorted(patch_dir.glob('X_pre_*.npy')),
        'flood': sorted(patch_dir.glob('X_flood_*.npy')),
        'dem': sorted(patch_dir.glob('X_dem_*.npy')),
        'mask': sorted(patch_dir.glob('y_mask_*.npy')),
        'full_mask': MODEL_READY_ROOT / str(year) / f'Assam_{year}_quick_flood_mask.npy',
    }

def verify_years(years):
    infos = []
    for year in years:
        paths = get_patch_paths(year)
        counts = {k: len(paths[k]) for k in ['pre', 'flood', 'dem', 'mask']}
        if not paths['patch_dir'].exists() or len(set(counts.values())) != 1 or counts['pre'] == 0:
            print(f'{year}: missing or mismatched chunks:', counts)
            continue

        chunk_counts = []
        for p in paths['mask']:
            chunk_counts.append(np.load(p, mmap_mode='r').shape[0])
        sample_pre = np.load(paths['pre'][0], mmap_mode='r')
        sample_dem = np.load(paths['dem'][0], mmap_mode='r')
        print(year, '| chunks:', counts['pre'], '| patch counts:', chunk_counts, '| full mask:', paths['full_mask'].exists())
        infos.append({
            'year': year,
            'paths': paths,
            'chunk_counts': chunk_counts,
            'count': sum(chunk_counts),
            'channels': sample_pre.shape[1],
            'dem_channels': sample_dem.shape[1],
        })
    return infos

year_infos = verify_years(YEARS)
AVAILABLE_YEARS = [info['year'] for info in year_infos]
if not year_infos:
    raise FileNotFoundError('No valid Assam patch chunks found. Run Assam_preprocessing.ipynb first.')
print('Available years:', AVAILABLE_YEARS)

## 8. Reconstruct Exact Patch Coordinates

`Assam_preprocessing.ipynb` did not save coordinate metadata, but it did save full quick masks. This cell replays the exact patch-selection logic:

```python
flood_fraction >= 0.01 or (flood_fraction == 0 and len(Y) % 5 == 0)
```

The detail that matters: `len(Y)` resets after every saved chunk, so this reconstruction also resets the per-chunk counter at `CHUNK_SIZE = 2000`.

In [ ]:
def reconstruct_coords_from_full_mask(year, expected_chunk_counts):
    full_mask_path = get_patch_paths(year)['full_mask']
    if not full_mask_path.exists():
        raise FileNotFoundError(f'Missing full mask for {year}: {full_mask_path}')

    mask = np.load(full_mask_path, mmap_mode='r')
    h, w = mask.shape
    coords_by_chunk = []
    current = []

    for row in range(0, h - PATCH_SIZE + 1, STRIDE):
        for col in range(0, w - PATCH_SIZE + 1, STRIDE):
            mask_patch = mask[row:row + PATCH_SIZE, col:col + PATCH_SIZE]
            flood_fraction = float(mask_patch.mean())
            keep_empty = (flood_fraction == 0 and len(current) % 5 == 0)

            if flood_fraction >= 0.01 or keep_empty:
                current.append((row // STRIDE, col // STRIDE, row, col))
                if len(current) >= CHUNK_SIZE:
                    coords_by_chunk.append(np.array(current, dtype=np.int32))
                    current = []

    if current:
        coords_by_chunk.append(np.array(current, dtype=np.int32))

    got = [len(c) for c in coords_by_chunk]
    if got != list(expected_chunk_counts):
        raise ValueError(f'{year}: reconstructed chunk counts {got}, expected {expected_chunk_counts}')
    return coords_by_chunk

coords_by_year = {}
for info in year_infos:
    coords_by_year[info['year']] = reconstruct_coords_from_full_mask(info['year'], info['chunk_counts'])
    print(info['year'], [c.shape for c in coords_by_year[info['year']]])

## 9. Build Node Features From Sentinel-1, DEM, Rainfall, and River Data

In [ ]:
def summarize_s1_patch(arr_chw):
    means = arr_chw.mean(axis=(1, 2))
    stds = arr_chw.std(axis=(1, 2))
    mins = arr_chw.min(axis=(1, 2))
    maxs = arr_chw.max(axis=(1, 2))
    return np.concatenate([means, stds, mins, maxs]).astype(np.float32)

def summarize_dem_patch(dem_chw):
    d = dem_chw[0]
    return np.array([d.mean(), d.std(), d.min(), d.max()], dtype=np.float32)

hydro_features = rain_features.merge(river_features, on='year', how='outer').fillna(0.0)
hydro_feature_cols = [c for c in hydro_features.columns if c != 'year']
hydro_by_year = hydro_features.set_index('year')[hydro_feature_cols].to_dict(orient='index')

MASK_POSITIVE_THRESHOLD = 0.02
all_x, all_y, all_meta = [], [], []

for info in year_infos:
    year = info['year']
    paths = info['paths']
    hydro = np.array([hydro_by_year.get(year, {}).get(c, 0.0) for c in hydro_feature_cols], dtype=np.float32)

    for chunk_i, (pre_p, flood_p, dem_p, mask_p) in enumerate(zip(paths['pre'], paths['flood'], paths['dem'], paths['mask'])):
        pre = np.load(pre_p, mmap_mode='r')
        flood = np.load(flood_p, mmap_mode='r')
        dem = np.load(dem_p, mmap_mode='r')
        mask = np.load(mask_p, mmap_mode='r')
        coords = coords_by_year[year][chunk_i]

        for local_i in tqdm(range(len(mask)), desc=f'features {year} chunk {chunk_i}'):
            pre_feat = summarize_s1_patch(pre[local_i])
            flood_feat = summarize_s1_patch(flood[local_i])
            change_feat = summarize_s1_patch(flood[local_i] - pre[local_i])
            dem_feat = summarize_dem_patch(dem[local_i])
            mask_fraction = float(mask[local_i, 0].mean())
            label = 1 if mask_fraction >= MASK_POSITIVE_THRESHOLD else 0
            grid_row, grid_col, pixel_row, pixel_col = coords[local_i]

            all_x.append(np.concatenate([pre_feat, flood_feat, change_feat, dem_feat, hydro]).astype(np.float32))
            all_y.append(label)
            all_meta.append({
                'year': year,
                'chunk': chunk_i,
                'local_patch': local_i,
                'grid_row': int(grid_row),
                'grid_col': int(grid_col),
                'pixel_row': int(pixel_row),
                'pixel_col': int(pixel_col),
                'mask_fraction': mask_fraction,
            })

X = np.vstack(all_x).astype(np.float32)
y = np.array(all_y, dtype=np.int64)
meta = pd.DataFrame(all_meta)
print('X:', X.shape)
print('y:', y.shape, 'positive fraction:', float(y.mean()))
meta.head()

## 10. Build GRU Temporal Sequences

In [ ]:
SEQUENCE_DAYS = 120
SEQUENCE_PRE_DAYS = 30
SEQ_FEATURE_COLS = ['rain_mean', 'rain_sum', 'river_mean', 'river_max']

def load_all_assam_rainfall():
    frames = []
    for year in YEARS:
        path = find_rainfall_file(year)
        if path is None:
            continue
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['Avg_rainfall'] = pd.to_numeric(df['Avg_rainfall'], errors='coerce')
        df = df.dropna(subset=['Date', 'Avg_rainfall'])
        frames.append(df[['Date', 'Avg_rainfall']])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['Date', 'Avg_rainfall'])

def load_all_assam_river():
    path = find_river_file()
    if path is None:
        return pd.DataFrame(columns=[RIVER_TIME_COL, RIVER_LEVEL_COL])
    df = pd.read_csv(path)
    df[RIVER_TIME_COL] = pd.to_datetime(df[RIVER_TIME_COL], errors='coerce', dayfirst=True)
    df[RIVER_LEVEL_COL] = pd.to_numeric(df[RIVER_LEVEL_COL], errors='coerce')
    return df.dropna(subset=[RIVER_TIME_COL, RIVER_LEVEL_COL])[[RIVER_TIME_COL, RIVER_LEVEL_COL]]

rain_all = load_all_assam_rainfall()
river_all = load_all_assam_river()

rain_daily = (
    rain_all.assign(day=rain_all['Date'].dt.floor('D'))
    .groupby('day')['Avg_rainfall']
    .agg(rain_mean='mean', rain_sum='sum')
    if len(rain_all) else pd.DataFrame(columns=['rain_mean', 'rain_sum'])
)
river_daily = (
    river_all.assign(day=river_all[RIVER_TIME_COL].dt.floor('D'))
    .groupby('day')[RIVER_LEVEL_COL]
    .agg(river_mean='mean', river_max='max')
    if len(river_all) else pd.DataFrame(columns=['river_mean', 'river_max'])
)

def build_year_sequence(year):
    event_start = pd.Timestamp(EVENT_DATES[year][0])
    seq_start = event_start - pd.Timedelta(days=SEQUENCE_PRE_DAYS)
    days = pd.date_range(seq_start, periods=SEQUENCE_DAYS, freq='D')
    seq_df = pd.DataFrame(index=days)
    seq_df.index.name = 'day'
    seq_df = seq_df.join(rain_daily, how='left').join(river_daily, how='left')
    return seq_df[SEQ_FEATURE_COLS].fillna(0.0).values.astype(np.float32)

year_sequences = {year: build_year_sequence(year) for year in AVAILABLE_YEARS}
for year, seq in year_sequences.items():
    print(year, seq.shape, 'rain_sum_total:', float(seq[:, 1].sum()), 'river_max:', float(seq[:, 3].max()))

seq_np = np.stack([year_sequences[int(year)] for year in meta['year'].values]).astype(np.float32)
print('Node temporal sequences:', seq_np.shape)

## 11. Build Exact Patch Graph Edges

In [ ]:
def build_edges(meta_df, neighbor_radius=1):
    lookup = {}
    for idx, row in meta_df.iterrows():
        lookup[(int(row['year']), int(row['grid_row']), int(row['grid_col']))] = idx

    edges = []
    for idx, row in meta_df.iterrows():
        year = int(row['year'])
        r = int(row['grid_row'])
        c = int(row['grid_col'])
        for dr in range(-neighbor_radius, neighbor_radius + 1):
            for dc in range(-neighbor_radius, neighbor_radius + 1):
                if dr == 0 and dc == 0:
                    continue
                j = lookup.get((year, r + dr, c + dc))
                if j is not None:
                    edges.append((idx, j))
    return np.array(edges, dtype=np.int64).T if edges else np.empty((2, 0), dtype=np.int64)

edge_index_np = build_edges(meta, neighbor_radius=1)
print('Nodes:', len(meta), 'Edges:', edge_index_np.shape[1], 'edge_index:', edge_index_np.shape)

## 12. Year-Holdout Split

In [ ]:
VAL_YEAR = 2023 if 2023 in AVAILABLE_YEARS else AVAILABLE_YEARS[-1]
train_mask_np = meta['year'].values != VAL_YEAR
val_mask_np = meta['year'].values == VAL_YEAR
print('Validation year:', VAL_YEAR)
print('Train nodes:', int(train_mask_np.sum()), 'positive:', float(y[train_mask_np].mean()))
print('Val nodes:', int(val_mask_np.sum()), 'positive:', float(y[val_mask_np].mean()))

## 13. Normalize Features and Move to Torch

In [ ]:
node_scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[train_mask_np] = node_scaler.fit_transform(X[train_mask_np])
X_scaled[val_mask_np] = node_scaler.transform(X[val_mask_np])

seq_scaler = StandardScaler()
seq_scaler.fit(seq_np[train_mask_np].reshape(-1, seq_np.shape[-1]))
seq_scaled = seq_scaler.transform(seq_np.reshape(-1, seq_np.shape[-1])).reshape(seq_np.shape).astype(np.float32)

x_t = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
seq_t = torch.tensor(seq_scaled, dtype=torch.float32, device=DEVICE)
y_t = torch.tensor(y, dtype=torch.long, device=DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long, device=DEVICE)
train_mask = torch.tensor(train_mask_np, dtype=torch.bool, device=DEVICE)
val_mask = torch.tensor(val_mask_np, dtype=torch.bool, device=DEVICE)

print('Node features:', x_t.shape)
print('Temporal sequences:', seq_t.shape)
print('Labels:', y_t.shape)
print('Edges:', edge_index.shape)

## 14. Define GCN + GRU Model

In [ ]:
def normalized_adjacency(edge_index, num_nodes, device):
    row, col = edge_index
    loops = torch.arange(num_nodes, device=device)
    row = torch.cat([row, loops])
    col = torch.cat([col, loops])
    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.clamp(min=1).pow(-0.5)
    values = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return torch.sparse_coo_tensor(torch.stack([row, col]), values, (num_nodes, num_nodes), device=device).coalesce()

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
    def forward(self, x, adj):
        return self.linear(torch.sparse.mm(adj, x))

class AssamGCNGRU(nn.Module):
    def __init__(self, node_in_dim, seq_in_dim, gcn_hidden=128, gru_hidden=64, fusion_hidden=128, dropout=0.25):
        super().__init__()
        self.gcn1 = GCNLayer(node_in_dim, gcn_hidden)
        self.gcn2 = GCNLayer(gcn_hidden, gcn_hidden)
        self.gru = nn.GRU(seq_in_dim, gru_hidden, batch_first=True, bidirectional=True)
        self.fusion = nn.Sequential(
            nn.Linear(gcn_hidden + 2 * gru_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 2),
        )
        self.dropout = dropout
    def forward(self, x, seq, adj):
        gx = F.relu(self.gcn1(x, adj))
        gx = F.dropout(gx, p=self.dropout, training=self.training)
        gx = F.relu(self.gcn2(gx, adj))
        _, h = self.gru(seq)
        hx = torch.cat([h[-2], h[-1]], dim=1)
        return self.fusion(torch.cat([gx, hx], dim=1))

adj = normalized_adjacency(edge_index, x_t.shape[0], DEVICE)
model = AssamGCNGRU(x_t.shape[1], seq_t.shape[2]).to(DEVICE)
print(model)

## 15. Train

In [ ]:
class_counts = np.bincount(y[train_mask_np], minlength=2)
class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = class_weights / class_weights.mean()
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print('Class counts:', class_counts, 'weights:', class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)

EPOCHS = 250
best_val_f1 = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(x_t, seq_t, adj)
    loss = criterion(logits[train_mask], y_t[train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(x_t, seq_t, adj)
        probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        preds = logits.argmax(dim=1).detach().cpu().numpy()

    train_f1 = precision_recall_fscore_support(y[train_mask_np], preds[train_mask_np], average='binary', zero_division=0)[2]
    val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(y[val_mask_np], preds[val_mask_np], average='binary', zero_division=0)
    val_acc = accuracy_score(y[val_mask_np], preds[val_mask_np])
    history.append({'epoch': epoch, 'loss': float(loss.item()), 'train_f1': float(train_f1), 'val_precision': float(val_precision), 'val_recall': float(val_recall), 'val_f1': float(val_f1), 'val_acc': float(val_acc)})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 25 == 0:
        print(f'Epoch {epoch:03d} | loss {loss.item():.4f} | train_f1 {train_f1:.3f} | val_f1 {val_f1:.3f} | val_acc {val_acc:.3f}')

model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print('Best validation F1:', best_val_f1)

## 16. Evaluate and Save

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x_t, seq_t, adj)
    prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
    pred = logits.argmax(dim=1).detach().cpu().numpy()

val_y = y[val_mask_np]
val_pred = pred[val_mask_np]
val_prob = prob[val_mask_np]
precision, recall, f1, _ = precision_recall_fscore_support(val_y, val_pred, average='binary', zero_division=0)
acc = accuracy_score(val_y, val_pred)
try:
    auc = roc_auc_score(val_y, val_prob)
except ValueError:
    auc = np.nan
metrics = {'val_year': int(VAL_YEAR), 'accuracy': float(acc), 'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'roc_auc': None if np.isnan(auc) else float(auc)}
print(metrics)
print(confusion_matrix(val_y, val_pred))

results = meta.copy()
results['target'] = y
results['pred'] = pred
results['prob_flood'] = prob
results.to_csv(OUTPUT_ROOT / 'assam_gnn_gru_patch_predictions.csv', index=False)
pd.DataFrame(history).to_csv(OUTPUT_ROOT / 'assam_gnn_gru_training_history.csv', index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_ROOT / 'assam_gnn_gru_metrics.csv', index=False)
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'GCN_GRU',
    'node_feature_dim': x_t.shape[1],
    'sequence_feature_dim': seq_t.shape[2],
    'sequence_days': SEQUENCE_DAYS,
    'sequence_feature_cols': SEQ_FEATURE_COLS,
    'hydro_feature_cols': hydro_feature_cols,
    'metrics': metrics,
}, OUTPUT_ROOT / 'assam_gnn_gru_model.pt')
print('Saved outputs to:', OUTPUT_ROOT)

## 17. Visualize Validation Year

In [ ]:
def plot_year_patch_map(results_df, year, value_col, title, cmap='viridis'):
    d = results_df[results_df['year'] == year]
    h = int(d['grid_row'].max()) + 1
    w = int(d['grid_col'].max()) + 1
    grid = np.full((h, w), np.nan, dtype=float)
    for _, row in d.iterrows():
        grid[int(row['grid_row']), int(row['grid_col'])] = row[value_col]
    plt.figure(figsize=(12, 6))
    plt.imshow(grid, cmap=cmap)
    plt.colorbar(label=value_col)
    plt.title(title)
    plt.axis('off')
    plt.show()

plot_year_patch_map(results, VAL_YEAR, 'target', f'{VAL_YEAR} pseudo target patches', cmap='Blues')
plot_year_patch_map(results, VAL_YEAR, 'prob_flood', f'{VAL_YEAR} GNN+GRU flood probability', cmap='magma')

## Notes

This notebook fixes the earlier path and coordinate issues:

- It auto-detects `Prj_3_Data` versus `PRJ3_DATA`.
- It uses the active Assam preprocessing event dates.
- It reconstructs exact patch coordinates from the full quick masks and the chunk-reset behavior.
- It holds out a full year for validation instead of using random patch validation.

The target is still the pseudo flood mask from Sentinel-1 change detection. If you later get verified flood labels, replace the `y_mask_*.npy` files or add a new target-loading section.